In [2]:
# Cell 1: Import libraries and setup
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, GatedGraphConv, LayerNorm
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
import h5py
import pickle
import math
import torch_geometric

# Initialize CUDA context explicitly
if torch.cuda.is_available():
    torch.cuda.init()
    print(f"CUDA initialized. Available GPUs: {torch.cuda.device_count()}")
else:
    print("CUDA not available, using CPU")

# Print torch_geometric version
print(f"torch_geometric version: {torch_geometric.__version__}")

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device for multi-GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
else:
    print("Using 1 GPU or CPU")

# Cell 2: Load adjacency matrix
def load_adjacency_matrix(adj_matrix_path):
    try:
        with open(adj_matrix_path, 'rb') as f:
            adj_data = pickle.load(f, encoding='latin1')

        if isinstance(adj_data, tuple):
            adj_matrix = next((item for item in adj_data if isinstance(item, np.ndarray)), None)
            if adj_matrix is None: raise ValueError("No numpy array found in the tuple.")
        elif isinstance(adj_data, dict):
            possible_keys = ['adj', 'adj_mx', 'adjacency_matrix', 'adj_matrix']
            key = next((k for k in possible_keys if k in adj_data), None)
            if key is None: raise ValueError(f"Adjacency matrix key not found in dict. Found keys: {adj_data.keys()}")
            adj_matrix = adj_data[key]
        else:
            adj_matrix = np.array(adj_data)

        adj_matrix = np.array(adj_matrix, dtype=np.float32)

        if adj_matrix.shape[0] == adj_matrix.shape[1]:
            adj_matrix = (adj_matrix + adj_matrix.T) / 2

        print(f"Loaded adjacency matrix from {adj_matrix_path}, shape: {adj_matrix.shape}")
        return adj_matrix

    except Exception as e:
        print(f"Error loading adjacency matrix from {adj_matrix_path}: {e}")
        raise

# Cell 3: Load PEMS-BAY data
def load_pems_bay_data(data_dir='PEMS-BAY', num_days=99, custom_adj_path=None):
    if custom_adj_path is None:
        raise ValueError("A path to a custom adjacency matrix must be provided for PEMS-BAY.")
    
    print(f"Loading custom adjacency matrix from: {custom_adj_path}")
    adj_mat = load_adjacency_matrix(custom_adj_path)
    
    with h5py.File(os.path.join(data_dir, 'PEMS-BAY.h5'), 'r') as f:
        node_values = np.array(f['speed']['block0_values'])

    time_steps = num_days * 288
    node_values = node_values[:time_steps].T
    print(f"Processed node_values shape: {node_values.shape}")

    scaler = MinMaxScaler()
    node_values_scaled = scaler.fit_transform(node_values.T).T
    
    edge_index = torch.tensor(np.where(adj_mat > 0), dtype=torch.long)
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)
    
    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)
    
    print(f"edge_index shape: {edge_index.shape}")
    print(f"edge_weight shape: {edge_weight.shape}")

    return node_values_scaled, edge_index, edge_weight, scaler

# Cell 4: Create data samples and 100th day data
def add_time_features(features, current_time_step_index, num_intervals_per_day=288, in_timestep=12):
    batch_size, current_in_timestep, num_nodes = features.shape
    time_features_list = []
    for i in range(current_in_timestep):
        global_time_step = (current_time_step_index + i) % num_intervals_per_day
        sin_time = math.sin(2 * math.pi * global_time_step / num_intervals_per_day)
        cos_time = math.cos(2 * math.pi * global_time_step / num_intervals_per_day)
        time_features = torch.tensor([[sin_time, cos_time]] * num_nodes, dtype=torch.float32)
        time_features_list.append(time_features)
    time_features_tensor = torch.stack(time_features_list, dim=0)
    time_features_batch = time_features_tensor.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    features_reshaped = features.unsqueeze(-1)
    features_with_time = torch.cat((features_reshaped, time_features_batch), dim=-1)
    return features_with_time

def create_data_samples(node_values, in_timestep=12, out_timestep=12, stride=1):
    features_list, labels_list = [], []
    num_nodes, total_time_steps = node_values.shape
    time_step_indices = range(0, total_time_steps - in_timestep - out_timestep + 1, stride)
    
    for i in time_step_indices:
        features_list.append(node_values[:, i:i + in_timestep])
        labels_list.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    
    if not features_list:
        return torch.empty(0, num_nodes, in_timestep * 3), torch.empty(0, num_nodes, out_timestep)

    features = torch.tensor(np.array(features_list, dtype=np.float32)).permute(0, 2, 1)
    labels = torch.tensor(np.array(labels_list, dtype=np.float32)).permute(0, 2, 1)
    
    features_with_time_list = [
        add_time_features(
            features[i].unsqueeze(0),
            global_start_time_step,
            in_timestep=in_timestep
        ).squeeze(0)
        for i, global_start_time_step in enumerate(time_step_indices)
    ]
    
    features_with_time = torch.stack(features_with_time_list, dim=0)
    features_final = features_with_time.permute(0, 2, 1, 3).reshape(features_with_time.shape[0], num_nodes, -1)
    return features_final, labels

def create_100th_day_data(data_dir='PEMS-BAY', in_timestep=12, out_timestep=12, stride=1):
    with h5py.File(os.path.join(data_dir, 'PEMS-BAY.h5'), 'r') as f:
        node_values_full = np.array(f['speed']['block0_values'])
        
    start_global_time_index = 288 * 99
    node_values = node_values_full[start_global_time_index : start_global_time_index + 288].T
    
    scaler_100 = MinMaxScaler()
    node_values_scaled = scaler_100.fit_transform(node_values.T).T
    
    features_final, labels = create_data_samples(node_values_scaled, in_timestep, out_timestep, stride)
    
    return features_final, labels, scaler_100

# Cell 5: Compute metrics and visualize predictions
def compute_metrics(pred, target, scaler):
    batch_size, num_nodes, out_timestep = pred.shape
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    mae = torch.mean(torch.abs(torch.tensor(pred_unscaled) - torch.tensor(target_unscaled))).item()
    rmse = torch.sqrt(torch.mean((torch.tensor(pred_unscaled) - torch.tensor(target_unscaled)) ** 2)).item()
    mask = torch.tensor(target_unscaled) > 1.0
    mape = torch.mean(torch.abs((torch.tensor(pred_unscaled)[mask] - torch.tensor(target_unscaled)[mask]) / torch.tensor(target_unscaled)[mask])) * 100
    return mae, rmse, mape.item()

def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5, filename='prediction.png'):
    os.makedirs(output_dir, exist_ok=True)
    pred_unscaled = scaler.inverse_transform(pred[0].cpu().numpy().T).T
    target_unscaled = scaler.inverse_transform(target[0].cpu().numpy().T).T
    
    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred.shape[1])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred_unscaled[i], label='Predicted', marker='o')
        plt.plot(target_unscaled[i], label='Actual', marker='x')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

# Cell 6: Define Graph Transformer model
class GraphTransformerLayer(nn.Module):
    def __init__(self, hidden_channels, heads=8, dropout=0.1):
        super(GraphTransformerLayer, self).__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=dropout)
        self.norm1 = LayerNorm(hidden_channels)
        self.norm2 = LayerNorm(hidden_channels)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight, batch):
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        x_gcn = self.norm1(x_gcn)
        x_trans = self.transformer(x_gcn, edge_index)
        x = x_gcn + x_trans + x
        x = self.norm2(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x_ffn = self.ffn(x)
        x = x + x_ffn
        return x

class GraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=12, num_layers=4, pe_dim=20, heads=4, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim if use_pe else hidden_channels)
        if use_pe:
            self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.layers = nn.ModuleList([GraphTransformerLayer(hidden_channels, heads=heads, dropout=0.1) for _ in range(num_layers)])
        self.norm = LayerNorm(hidden_channels)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )
    
    def forward(self, x, edge_index, edge_weight, pe=None, batch=None):
        x = self.node_emb(x)
        if self.use_pe and pe is not None:
            pe = self.pe_lin(pe)
            x = torch.cat([x, pe], dim=-1)
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
        x = self.norm(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1 if batch is not None else 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)

# Cell 7: Main workflow
adj_matrices = [
    'adj_mx_knn_lane_pems.pkl',
    'adj_mx_knn_pems.pkl',
    'adj_mx_radius_lane_pems.pkl',
    'adj_mx_radius_pems.pkl'
]
stride_options = {
    '1': ('5 minutes', 1),
    '2': ('1 hour', 12),
    '3': ('1 day', 288)
}
results = {}

for adj_file in adj_matrices:
    for stride_key, (stride_name, stride_val) in stride_options.items():
        print(f"\n--- Processing Adjacency Matrix: {adj_file}, Stride: {stride_name} ({stride_val} time steps) ---")
        
        try:
            node_values, edge_index, edge_weight, scaler = load_pems_bay_data(custom_adj_path=f'PEMS-BAY/{adj_file}')
            in_timestep, out_timestep = 12, 12

            features, labels = create_data_samples(node_values, in_timestep=in_timestep, out_timestep=out_timestep, stride=stride_val)
            
            if features.nelement() == 0:
                print("No samples generated with current configuration. Skipping.")
                continue

            total_samples = len(features)
            train_end = int(0.6 * total_samples)
            val_end = train_end + int(0.2 * total_samples)
            
            train_features, train_labels = features[:train_end], labels[:train_end]
            val_features, val_labels = features[train_end:val_end], labels[train_end:val_end]
            test_features, test_labels = features[val_end:], labels[val_end:]
            
            print(f"Train samples: {len(train_features)}, Val samples: {len(val_features)}, Test samples: {len(test_features)}")
            
            transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
            
            def create_pyg_dataset(features, labels, edge_index, edge_weight):
                return [transform(Data(x=f, y=l.T, edge_index=edge_index, edge_attr=edge_weight)) for f, l in zip(features, labels)]

            train_data = create_pyg_dataset(train_features, train_labels, edge_index, edge_weight)
            val_data = create_pyg_dataset(val_features, val_labels, edge_index, edge_weight)
            test_data = create_pyg_dataset(test_features, test_labels, edge_index, edge_weight)

            day_100_features, day_100_labels, day_100_scaler = create_100th_day_data(in_timestep=in_timestep, out_timestep=out_timestep, stride=1)
            day_100_data = create_pyg_dataset(day_100_features, day_100_labels, edge_index, edge_weight)
            
            train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
            val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
            test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
            day_100_loader = DataLoader(day_100_data, batch_size=32, shuffle=False)
            
            model_in_channels = in_timestep * 3
            model = GraphTransformer(in_channels=model_in_channels, hidden_channels=64, out_channels=out_timestep).to(device)
            
            optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
            criterion = nn.MSELoss()
            
            for epoch in range(20):
                model.train()
                total_train_loss = 0
                for data in train_loader:
                    data = data.to(device)
                    optimizer.zero_grad()
                    out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
                    target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                    loss = criterion(out, target)
                    loss.backward()
                    optimizer.step()
                    total_train_loss += loss.item()
                
                val_loss = float('nan')
                if len(val_loader) > 0:
                    model.eval()
                    total_val_loss = 0
                    with torch.no_grad():
                        for data in val_loader:
                            data = data.to(device)
                            out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
                            target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                            total_val_loss += criterion(out, target).item()
                    val_loss = total_val_loss / len(val_loader)

                print(f"Epoch {epoch + 1}, Train Loss: {total_train_loss / len(train_loader):.4f}, Val Loss: {val_loss:.4f}")

            def evaluate(loader, scaler_obj, set_name):
                model.eval()
                total_mae, total_rmse, total_mape, num_batches = 0, 0, 0, 0
                if len(loader) == 0: return {'mae': 0, 'rmse': 0, 'mape': 0}
                with torch.no_grad():
                    for i, data in enumerate(loader):
                        data = data.to(device)
                        out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
                        target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                        mae, rmse, mape = compute_metrics(out, target, scaler_obj)
                        total_mae += mae
                        total_rmse += rmse
                        total_mape += mape
                        num_batches += 1
                        if i == 0: visualize_predictions(out, target, scaler_obj, output_dir=f'plots_{adj_file.replace(".pkl", "")}', filename=f'preds_{set_name}_{stride_val}.png')
                
                avg_mae = total_mae / num_batches
                avg_rmse = total_rmse / num_batches
                avg_mape = total_mape / num_batches
                print(f"{set_name} - MAE: {avg_mae:.4f}, RMSE: {avg_rmse:.4f}, MAPE: {avg_mape:.4f}%")
                return {'mae': avg_mae, 'rmse': avg_rmse, 'mape': avg_mape}

            print("\n--- Evaluating Test Set ---")
            test_metrics = evaluate(test_loader, scaler, "Test")
            print("\n--- Evaluating 100th Day Set ---")
            day_100_metrics = evaluate(day_100_loader, day_100_scaler, "100th Day")
            
            results[f"{adj_file}_stride_{stride_val}"] = {'test': test_metrics, '100th_day': day_100_metrics}

        except Exception as e:
            print(f"Error processing {adj_file} with stride {stride_val}: {e}")
            results[f"{adj_file}_stride_{stride_val}"] = {
                'test': {'mae': float('nan'), 'rmse': float('nan'), 'mape': float('nan')},
                '100th_day': {'mae': float('nan'), 'rmse': float('nan'), 'mape': float('nan')}
            }

# Cell 8: Print summary of results
print("\n\n--- Final Summary of All Results ---")
for key, metrics in results.items():
    adj_file_name, stride_info = key.split('_stride_')
    print(f"\nAdjacency Matrix: {adj_file_name}, Stride: {stride_info}")
    if metrics.get('test'):
      print(f"  Test      - MAE: {metrics['test']['mae']:.4f} mph, RMSE: {metrics['test']['rmse']:.4f} mph, MAPE: {metrics['test']['mape']:.4f}%")
    if metrics.get('100th_day'):
      print(f"  100th Day - MAE: {metrics['100th_day']['mae']:.4f} mph, RMSE: {metrics['100th_day']['rmse']:.4f} mph, MAPE: {metrics['100th_day']['mape']:.4f}%")

CUDA initialized. Available GPUs: 2
torch_geometric version: 2.6.1
Using 2 GPUs!

--- Processing Adjacency Matrix: adj_mx_knn_lane_pems.pkl, Stride: 5 minutes (1 time steps) ---
Loading custom adjacency matrix from: PEMS-BAY/adj_mx_knn_lane_pems.pkl
Loaded adjacency matrix from PEMS-BAY/adj_mx_knn_lane_pems.pkl, shape: (325, 325)
Processed node_values shape: (325, 28512)
edge_index shape: torch.Size([2, 1136])
edge_weight shape: torch.Size([1136])
Train samples: 17093, Val samples: 5697, Test samples: 5699
Epoch 1, Train Loss: 0.0170, Val Loss: 0.0088
Epoch 2, Train Loss: 0.0054, Val Loss: 0.0080
Epoch 3, Train Loss: 0.0048, Val Loss: 0.0071
Epoch 4, Train Loss: 0.0046, Val Loss: 0.0081
Epoch 5, Train Loss: 0.0046, Val Loss: 0.0065
Epoch 6, Train Loss: 0.0044, Val Loss: 0.0064
Epoch 7, Train Loss: 0.0044, Val Loss: 0.0079
Epoch 8, Train Loss: 0.0043, Val Loss: 0.0073
Epoch 9, Train Loss: 0.0043, Val Loss: 0.0072
Epoch 10, Train Loss: 0.0042, Val Loss: 0.0086
Epoch 11, Train Loss: 0.004